# Roof + circulation

A path people walk on, over any roof: a flat one, or one as irregular as rock. You draw where it goes
in plan; the module finds the roof's surface under it and lays a walking level on it that is never
steeper than 1:12, with **no stairs**. It hands back a `Contribution`: solid to **add** (the path,
reaching down into the roof), space to **cut** (headroom over it), and solid to **keep** (under the
walking surface, safe from later processes). `combine` puts it on the roof, alongside the other modules.

Each step below is one group of components on a Grasshopper canvas, written as code. The logic lives in
`roof_circulation/`, where it is tested. Units are millimetres.

In [ ]:
%matplotlib inline
import roof_circulation as work

## Step 1 · The brief
*Canvas: number sliders*

The rules every path keeps are fixed or bounded: a ramp is never steeper than 1:12 (`ramp_max` may be
gentler, never steeper), a level landing `width / 2` long at every turn and end, `headroom` kept clear above,
`support` protected below. What you tune: `width`, how closely the path follows the roof (`smoothing`), and
the wave along it (`wave`, `wavelength`, any length), how far it rides above the rock (`lift`), and how
far its edge keeps from the roof's edge and holes, all along it (`edge_clearance`). The rest sizes the stand-in rocky roof.

In [ ]:
brief = work.Brief(width=1500, edge_clearance=500, smoothing=3000, lift=10, wave=60, wavelength=6000,
                   avoid_slope=4, turns=4, swing=0.5,
                   bend=2400, attract=0.7, repel=9, influence=3000, roughness=400)

## Step 2 · The roof's slopes
*Canvas: a mesh coloured by slope*

The stand-in roof is a rocky slab; any field can take its place, a mesh through `VolGrid.from_mesh`. Read
every `cell` mm, its slope shows where a path could follow it as it is (green), and where it would have to
be cut in or built up (red). Holes, such as another module's openings, are white.

**Try:** `roughness=200` for a gentler rock, or `rock_seed=7` for another one.

In [ ]:
roof = work.steps.host(brief)
found = work.steps.slopes(brief, roof)
work.preview.slope_map(found, brief.ramp_max);

## Step 3 · The route
*Canvas: **Shortest Walk** over the slope map*

You give a **start** and an **end**, here one corner of the roof to the other. `route` finds the way between
them over the slopes: a step costs more the steeper it climbs *in the direction it goes*, and more the further
it turns, so the route keeps to gentle ground and crosses slopes along their contours, in long smooth legs.
`turns` sets how many times it swings side to side on the way: 0 goes as straight as the slope allows, more
winds more; `swing` how far each swing goes aside. Each swing settles on the gentlest ground near it, and the
whole route is one smooth curve, its bends smoothed over `bend` mm. **Attractors** (green) draw it towards a
place, a view or a planter: one within `influence` of a swing captures it, the route swings to it; otherwise it
only makes nearby ground cheaper, a pull steep ground can win against. **Repellers** (purple) keep it away, from a
drain or an opening's edge. Other modules' features can be given as either.

**Try:** `turns=0`, `turns=2`, `turns=8`; `swing=1` for wide bows; `bend=600` to hug the ground, `bend=9600`
to sweep. Move the attractor and the repeller, or add more. Or give only an entrance, `Drawing(entrances=[...])`, and call
`network(brief, roof, found, drawing)` to grow `branches` paths out over the gentle ground instead.

In [ ]:
from compas.geometry import Point

drawing = work.Drawing.sample_crossing()
drawing.attractors = [Point(6000, 6000, 0)]
drawing.repellers = [Point(11000, 8100, 0)]
drawing = work.steps.route(brief, found, drawing)
work.preview.slope_map(found, brief.ramp_max, drawing);

### Or draw the path yourself
The third way to make a path, beside `route` and `network`: draw its centre line in plan with the corners you
want, and pin the level wherever you want one (red), here a lookout held at 1300. The rest of the steps treat it
like any other path. **Try:** to carry on with it, run `drawing = drawn` and go on from Step 4.

In [ ]:
drawn = work.Drawing.sample()
work.preview.slope_map(found, brief.ramp_max, drawn);

## Step 4 · The grade
*Canvas: **Graph Mapper**, in effect*

The level follows the roof averaged over `smoothing`, plus the wave fading out at the ends. It meets the roof
at both ends, keeps every pin, is level through every turn, and never passes 1:12. Grey is the roof under the
path; blue is where you walk. Where the line leaves the grey, the path is cut into the roof or built up on it.

Every fork and end is held level as far as the paths meeting there overlap, so they meet without a step.

In [ ]:
profiles = work.steps.grade(brief, work.steps.ground(brief, roof, drawing))
print("cut and fill (mm):", work.checks.cut_and_fill(profiles))
work.preview.profile(profiles);

## Step 4b · Rest landings (optional)
*Canvas: its own component, with its own numbers: leave it out for no rests*

A path may climb at most `rise` (750 by code) before a level **rest landing** `length` long, with a bay `bay`
deep beside it to sit in or turn a wheelchair. These numbers are a `RestBrief` of their own, apart from the
brief: the path's width and the rest are set in Step 1. Skip this cell and the path has no rests. Climb is
counted along one continuous climb, and starts again where the path turns down or rests. Each rest goes by an
attractor if one is near, else on the flattest ground; green on the profile.

In [ ]:
landing = work.RestBrief(rise=750, length=1500, bay=1500)
profiles = work.steps.rest(brief, landing, roof, profiles, drawing)
print(sum(len(p.rests) for p in profiles), "rest landings on this route")

The route above keeps to level ground and never climbs 750, so here is one that must: across the roof by a
lookout held at 850, resting every 300 to show it.

**Try:** `rise=200` for more; `bay=0` for no bays; `length=3000` and see the climb run out of room.

In [ ]:
climb = work.Brief(**{**brief.to_dict(), "turns": 0})
lookout = work.steps.route(climb, found, work.Drawing.sample_lookout())
lookout_profiles = work.steps.grade(climb, work.steps.ground(climb, roof, lookout))
rested = work.steps.rest(climb, work.RestBrief(rise=300), roof, lookout_profiles, lookout)
print(len(rested[0].rests), "rest landings")
work.preview.profile(rested);

## Step 5 · The module, on the roof
*Canvas: **Volumetric Boolean***

`circulation` is the whole module. `combine` adds every module's solid to the roof, then cuts every module's
space: where the path runs below the rock it opens a cutting, where above it stands on its own fill.

In [ ]:
runs = work.steps.segments(brief, profiles)
bounds = work.steps.extent(brief, profiles)
part = work.steps.circulation(brief, runs)
solid = work.combine(roof, [part])
work.preview.section(bounds, solid, axis="y", at=2000, title="across the entrance")
work.preview.section(bounds, solid, axis="x", at=10000, title="across the middle");

## Step 6 · Checks
The rules measured on the finished solid: nothing steeper than 1:12, and at every sample floor underfoot and
nothing overhead, not even a 10 mm lip. Run this again after other modules are combined.

In [ ]:
print("steepest: 1:{:.1f}".format(1 / work.checks.steepest(runs)))
print(work.checks.path_clear(brief, runs, solid))

## Step 7 · Export
*Canvas: **Volumetric Mesh***

Writes the mesh (`.stl`) and the field (`.json`) to `out/`, which git ignores. Lower `voxel` for a finer mesh.

In [ ]:
vertices, faces = work.steps.export(brief, solid, bounds, "out/roof_circulation")
print("{:,} triangles".format(len(faces)))